# NMI Forecastability Score and Classification

The workflow includes:

1. Identifying the active window for each NMI instead of enforcing a common start date.
2. Calculating metrics for data quality, history coverage, seasonality, temporal dependence, and stability.
3. Evaluating baseline forecasting performance using backtesting and WAPE.
4. Standardising metrics into 0–1 component scores and combining them into an overall forecastability score.
5. Producing final classifications and recommended modelling strategies.

## 0. Import libraries and set paths

In [12]:
from __future__ import annotations

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

# Keep notebook outputs tidy
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

# Main project folders
PROJECT_ROOT = Path("/Users/chenlinchao/Desktop/Energy_Forecasting_Group_2")
DATA_DIR = PROJECT_ROOT / "Datasets"
EDA_DIR = PROJECT_ROOT / "EDA_Tasks"

# Set folder for the score/classification outputs
OUTPUT_DIR = EDA_DIR / "NMI_score_classification_results"
OUTPUT_DIR.mkdir(exist_ok=True)

# Input files used in this notebook.
ENERGY_PARQUET = DATA_DIR / "Clariti Consumption/LMS_2013-01-01_2026-03-24_HALF_HOUR_au.pq"
ENERGY_CSV = DATA_DIR / "Clariti Consumption/LMS_2013-01-01_2026-03-24_HALF_HOUR_au.csv"
ARCHIBUS_XLSX = DATA_DIR / "Archibus Extract_Buildings_May 2026.xlsx"
LMS_NMI_MAP_XLSX = DATA_DIR / "LMS Serial to NMI Map.xlsx"
PARKVILLE_SUBSTATION_XLSX = DATA_DIR / "Parkville Substation Mapping.xlsx"
BUILDING_NMI_JSON = EDA_DIR / "4/building_nmi_mapping.json"

# NMIs excluded based on earlier EDA checks.
DROP_NMIS = {"6102507141", "VAAA003225"}

# Backtesting settings: at least one year of history and about one month of validation data
MIN_ACTIVE_DAYS_FOR_BACKTEST = 365
MIN_VALIDATION_POINTS = 48 * 30


print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: /Users/chenlinchao/Desktop/Energy_Forecasting_Group_2
OUTPUT_DIR: /Users/chenlinchao/Desktop/Energy_Forecasting_Group_2/EDA_Tasks/NMI_score_classification_results


## 1. Load and Clean Energy Consumption Data


`6102002531 consumption` -> `6102002531`

In [13]:
# Load half-hourly energy data.
energy_raw = pd.read_parquet(ENERGY_PARQUET)

# Make sure the records are ordered by time.
energy_raw["date"] = pd.to_datetime(energy_raw["date"])
energy_raw = energy_raw.sort_values("date").reset_index(drop=True)

# Use the NMI itself as the column name, e.g. "6102002531 consumption" -> "6102002531".
energy = energy_raw.copy()
energy.columns = [
    col.replace(" consumption", "") if col != "date" else col
    for col in energy.columns
]

# Drop NMIs removed in the earlier EDA, if they appear in this dataset.
existing_drop_cols = [nmi for nmi in DROP_NMIS if nmi in energy.columns]
energy = energy.drop(columns=existing_drop_cols)

# Store the NMI list and the latest timestamp for later metric calculation.
nmi_cols = [col for col in energy.columns if col != "date"]
latest_timestamp = energy["date"].max()

print("Energy table shape:", energy.shape)
print("Number of NMI columns:", len(nmi_cols))
print("Date range:", energy["date"].min(), "to", latest_timestamp)
print("Dropped NMIs:", existing_drop_cols)

energy.head()


Energy table shape: (231840, 100)
Number of NMI columns: 99
Date range: 2013-01-01 00:00:00 to 2026-03-23 23:30:00
Dropped NMIs: ['6102507141', 'VAAA003225']


,date,6102000812,6102002302,6102005454,6102005592,6102009742,6102009743,6102009744,6102023971,6102038376,6102046251,6102047562,6102079015,6102126219,6102136796,6102159807,6102241120,6102253330,6102284820,6102329966,6102332526,6102479831,6102548873,6102573328,6102741362,6102798810,6102802017,6102823324,6103002422,6103004482,6103005867,6103005869,6103009639,6103010081,6103010326,6103011168,6103015873,6103022015,6103022017,6103022018,6103029662,6103029663,6103031269,6103031796,6103054578,6103054611,6103055142,6103055392,6103055643,6103056620,6103056621,6103056622,6103056625,6103063019,6103063020,6103065120,6103065121,6103065471,6103066694,6103067996,6103068525,6103077259,6103086897,6203397519,6203397522,6203848319,6203949247,VAAA000057,VAAA000173,VAAA000175,VAAA000177,VAAA000178,VAAA000180,VAAA000181,VAAA000182,VAAA000184,VAAA000185,VAAA000237,VAAA000450,VAAA000507,VAAA000936,VAAA001188,VAAA001259,VAAA001271,VAAA001329,VAAA001356,VAAA001472,VAAA001492,VAAA002242,VAAA002408,VAAA003100,VAAA003176,VAAA003194,VAAA003197,VAAA003429,VAAA004066,VCCCAE0035,VCCCBC0096,VCCCSC0045,VCCCSD0058
0,2013-01-01 00:00:00,20.768,11.659,24.76,17.92,5.232,18.735,13.199,204.30,70.84,17.60,26.24,4.18,0.0,0.0,0.0,4.432,87.30,1.432,23.334,29.28,6.400,184.50,12.120,0.0,0.0,0.0,152.588,141.360,119.640,101.280,140.88,0.0,0.0,0.0,0.0,352.079,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,58.980,63.719,236.173,49.120,72.448,42.445,54.106,333.998,319.195,70.993,47.595,29.437,59.52,0.0,4.512,0.0,0.0,0.0,0.0,14.834,3.816,11.26,5.456,188.424,16.74,19.76,28.20,0.0,0.0,95.968,23.490,9.320,21.087
1,2013-01-01 00:30:00,20.512,11.260,24.64,18.52,5.376,18.700,13.020,207.66,70.56,16.96,0.96,4.22,0.0,0.0,0.0,4.256,87.96,1.559,25.095,29.16,5.759,183.00,11.280,0.0,0.0,0.0,153.283,140.759,123.720,100.199,138.36,0.0,0.0,0.0,0.0,353.520,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,57.180,62.878,236.439,49.931,74.850,42.849,55.842,330.608,311.678,69.793,47.836,29.672,61.46,0.0,4.832,0.0,0.0,0.0,0.0,14.912,3.344,11.24,5.964,187.908,17.76,19.82,28.72,0.0,0.0,96.512,23.865,9.072,19.296
2,2013-01-01 01:00:00,20.544,11.459,24.92,18.20,4.720,18.644,13.620,205.68,69.52,17.24,0.88,4.22,0.0,0.0,0.0,4.568,87.84,1.504,25.608,26.88,5.120,181.14,9.720,0.0,0.0,0.0,155.346,139.439,124.199,102.480,137.76,0.0,0.0,0.0,0.0,353.099,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.840,65.380,233.507,50.631,71.847,42.208,54.891,333.359,320.405,69.330,47.771,29.384,61.00,0.0,4.480,0.0,0.0,0.0,0.0,14.850,3.884,11.28,6.332,187.500,17.46,19.68,28.60,0.0,0.0,95.423,24.165,9.080,19.904
3,2013-01-01 01:30:00,20.688,11.419,24.56,17.72,3.582,17.130,13.260,202.32,70.12,17.32,0.96,4.32,0.0,0.0,0.0,4.904,88.62,1.584,25.344,25.98,5.280,180.24,10.079,0.0,0.0,0.0,153.114,141.599,117.720,97.800,135.84,0.0,0.0,0.0,0.0,354.480,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,57.119,64.219,242.771,50.141,74.661,42.737,54.340,329.039,318.910,70.418,48.669,29.376,61.24,0.0,4.448,0.0,0.0,0.0,0.0,15.006,5.708,11.80,6.448,187.180,17.10,19.76,28.26,0.0,0.0,98.783,22.380,9.072,19.552
4,2013-01-01 02:00:00,22.032,11.340,24.64,18.92,3.952,17.155,12.899,206.82,92.72,17.32,0.96,4.36,0.0,0.0,0.0,4.216,91.68,1.543,25.517,25.62,5.408,175.62,12.240,0.0,0.0,0.0,154.387,139.559,121.560,84.240,121.80,0.0,0.0,0.0,0.0,350.760,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,56.759,63.758,232.918,49.703,71.184,42.435,55.810,328.735,311.482,70.164,47.074,29.835,60.44,0.0,4.768,0.0,0.0,0.0,0.0,15.019,3.292,11.82,5.704,187.013,18.00,19.74,27.70,0.0,0.0,98.016,22.995,9.216,20.032


## 2. Load Building Mapping and Building Type

In [14]:
# Building and NMI mapping functions
def normalise_code(value):
    # Convert building codes, NMIs, and other keys into clean strings
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text


def unique_join(values, sep="; ") -> str:
    # Deduplicate, sort, and join a group of values into one string
    clean = sorted({str(v).strip() for v in values if pd.notna(v) and str(v).strip() != ""})
    return sep.join(clean) if clean else pd.NA


# Load Archibus building metadata
archibus = pd.read_excel(ARCHIBUS_XLSX, header=2)
archibus = archibus.rename(columns={
    "Building Code": "building_code",
    "Building Name": "building_name",
    "Building Type": "building_type",
    "Campus Code": "campus_code",
    "Longitude": "longitude",
    "Latitude": "latitude",
})
archibus["building_code"] = archibus["building_code"].map(normalise_code)

# The Archibus Building Type field contains a few obvious dirty values, such as addresses or .N/A
# Keep only reliable type labels; unreliable values become missing and are later shown as Unknown
def clean_building_type(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    bad_values = {"", ".N/A", "N/A", "NA", "nan", "none", "c"}
    if text in bad_values or text[:1].isdigit():
        return pd.NA
    return text

archibus["building_type"] = archibus["building_type"].map(clean_building_type)

archibus_lookup = archibus[[
    "building_code", "building_name", "building_type", "campus_code", "longitude", "latitude"
]].drop_duplicates()

print("Archibus buildings:", archibus_lookup.shape)
archibus_lookup.head()

Archibus buildings: (871, 6)


,building_code,building_name,building_type,campus_code,longitude,latitude
0,000BAL,BALLARAT - GROUNDS,Land,COM,NaN,NaN
1,000BUR,BURNLEY CAMPUS - GROUNDS,Land,BUR,NaN,NaN
2,000CRE,CRESWICK CAMPUS - GROUNDS,Land,CRE,NaN,NaN
3,000DOO,DOOKIE CAMPUS - GROUNDS,Land,DOO,NaN,NaN
4,000FSB,FISHERMANS BEND SITE - GROUNDS,Land,FSB,144.922239,-37.827261


In [15]:
# Build the NMI-to-building mapping table

mapping_rows = []

# Read mappings from building_nmi_mapping.json if available
if BUILDING_NMI_JSON.exists():
    with BUILDING_NMI_JSON.open() as f:
        building_mapping = json.load(f)

    for section, entries in building_mapping.items():
        if section == "unmapped_nmis":
            for nmi in entries:
                mapping_rows.append({
                    "NMI": normalise_code(nmi),
                    "building_code": pd.NA,
                    "mapping_source": "building_nmi_mapping_json",
                    "mapping_section": "unmapped_nmis",
                })
        else:
            for building_code, nmis in entries.items():
                for nmi in nmis:
                    mapping_rows.append({
                        "NMI": normalise_code(nmi),
                        "building_code": normalise_code(building_code),
                        "mapping_source": "building_nmi_mapping_json",
                        "mapping_section": section,
                    })

# Read direct building codes from LMS Serial to NMI Map
# When LocationCode looks like PAR;104, the part after the semicolon is usually the building code.
if LMS_NMI_MAP_XLSX.exists():
    lms_map = pd.read_excel(LMS_NMI_MAP_XLSX)
    lms_map["NMI"] = lms_map["NMI"].map(normalise_code)
    lms_map["LocationCode"] = lms_map["LocationCode"].map(normalise_code)

    for _, row in lms_map.iterrows():
        location = row.get("LocationCode")
        building_code = pd.NA
        if isinstance(location, str) and ";" in location:
            building_code = normalise_code(location.split(";")[-1])
        mapping_rows.append({
            "NMI": row["NMI"],
            "building_code": building_code,
            "mapping_source": "LMS Serial to NMI Map",
            "mapping_section": "location_code",
        })

# Read shared/substation building mappings from Parkville Substation Mapping
# The original file has title rows, so manually read columns B-F and forward-fill substation/NMI/split fields
if PARKVILLE_SUBSTATION_XLSX.exists():
    parkville_raw = pd.read_excel(PARKVILLE_SUBSTATION_XLSX, sheet_name="Parkville", header=None)
    parkville = parkville_raw.iloc[5:, 1:6].copy()
    parkville.columns = ["substation", "NMI", "meter_split", "building_code", "building_name_from_substation"]
    parkville[["substation", "NMI", "meter_split"]] = parkville[["substation", "NMI", "meter_split"]].ffill()
    parkville["NMI"] = parkville["NMI"].map(normalise_code)
    parkville["building_code"] = parkville["building_code"].map(normalise_code)
    parkville = parkville.dropna(subset=["NMI"])

    for _, row in parkville.iterrows():
        mapping_rows.append({
            "NMI": row["NMI"],
            "building_code": row["building_code"],
            "mapping_source": "Parkville Substation Mapping",
            "mapping_section": "substation_shared",
        })

nmi_building_long = pd.DataFrame(mapping_rows).drop_duplicates()
nmi_building_long = nmi_building_long[nmi_building_long["NMI"].notna()]

# Merge with Archibus to attach building type, name, and campus.
nmi_building_enriched = nmi_building_long.merge(archibus_lookup, on="building_code", how="left")

print("Long NMI-building mapping shape:", nmi_building_enriched.shape)
nmi_building_enriched.head(10)

Long NMI-building mapping shape: (332, 9)


,NMI,building_code,mapping_source,mapping_section,building_name,building_type,campus_code,longitude,latitude
0,6102332526,104,building_nmi_mapping_json,building_to_nmis,ALAN GILBERT BUILDING,Building - Workspace/Teaching/Library,PAR,144.959137,-37.800217
1,6103029662,104,building_nmi_mapping_json,building_to_nmis,ALAN GILBERT BUILDING,Building - Workspace/Teaching/Library,PAR,144.959137,-37.800217
2,6103029663,104,building_nmi_mapping_json,building_to_nmis,ALAN GILBERT BUILDING,Building - Workspace/Teaching/Library,PAR,144.959137,-37.800217
3,6102823324,105,building_nmi_mapping_json,building_to_nmis,FBE BUILDING,Building - Workspace/Teaching/Library,PAR,144.959230,-37.801562
4,6102253330,106,building_nmi_mapping_json,building_to_nmis,LAW BUILDING,Building - Workspace/Teaching/Library,PAR,144.960031,-37.802422
5,6102573328,107,building_nmi_mapping_json,building_to_nmis,UNIVERSITY SQUARE CARPARK,Building - Commercial,COM,144.960376,-37.800483
6,6102005454,110,building_nmi_mapping_json,building_to_nmis,THE SPOT,Building - Workspace/Teaching/Library,PAR,144.958797,-37.801606
7,6102005592,110,building_nmi_mapping_json,building_to_nmis,THE SPOT,Building - Workspace/Teaching/Library,PAR,144.958797,-37.801606
8,VAAA000507,130,building_nmi_mapping_json,building_to_nmis,PARKVILLE BUILDING 130,Building - Commercial,COM,144.960904,-37.796877
9,6103005867,144,building_nmi_mapping_json,building_to_nmis,KENNETH MYER BUILDING,Building - Highly Serviced / Wet Lab,PAR,144.958549,-37.798258


In [16]:
# Summarise building information into one row per NMI

def mapping_quality_for_group(group: pd.DataFrame) -> str:
    # Create a mapping quality label for each NMI based on mapping source and number of buildings
    sections = set(group["mapping_section"].dropna().astype(str))
    building_codes = set(group["building_code"].dropna().astype(str))

    if "unmapped_nmis" in sections and not building_codes:
        return "unmapped"
    if "many_to_many" in sections:
        return "many_to_many"
    if "many_to_one" in sections:
        return "many_to_one"
    if "substation_shared" in sections and len(building_codes) > 1:
        return "substation_shared_multi_building"
    if len(building_codes) == 1:
        return "one_building_mapped"
    if len(building_codes) > 1:
        return "multi_building_mapped"
    return "unknown"


def building_type_group(types_text) -> str:
    # Classify building type coverage as Single, Mixed, or Unknown
    if pd.isna(types_text) or str(types_text).strip() == "":
        return "Unknown"
    types = [x.strip() for x in str(types_text).split(";") if x.strip()]
    if len(set(types)) <= 1:
        return "Single"
    return "Mixed"


if nmi_building_enriched.empty:
    nmi_metadata = pd.DataFrame({"NMI": nmi_cols})
else:
    nmi_metadata = (
        nmi_building_enriched
        .groupby("NMI")
        .agg(
            building_codes=("building_code", unique_join),
            building_names=("building_name", unique_join),
            building_type=("building_type", unique_join),
            campus_codes=("campus_code", unique_join),
            mapping_sources=("mapping_source", unique_join),
            mapping_quality=("mapping_section", lambda _: pd.NA),
        )
        .reset_index()
    )

    quality = (
        nmi_building_enriched.groupby("NMI")
        .apply(mapping_quality_for_group)
        .rename("mapping_quality")
        .reset_index()
    )
    nmi_metadata = nmi_metadata.drop(columns=["mapping_quality"]).merge(quality, on="NMI", how="left")

nmi_metadata["building_type_group"] = nmi_metadata["building_type"].apply(building_type_group)
nmi_metadata["building_type_count"] = nmi_metadata["building_type"].apply(
    lambda x: 0 if pd.isna(x) else len({v.strip() for v in str(x).split(";") if v.strip()})
)

# Keep only NMIs that exist in the energy dataset
nmi_metadata = pd.DataFrame({"NMI": nmi_cols}).merge(nmi_metadata, on="NMI", how="left")
nmi_metadata["building_type"] = nmi_metadata["building_type"].fillna("Unknown")
nmi_metadata["building_type_group"] = nmi_metadata["building_type_group"].fillna("Unknown")
nmi_metadata["mapping_quality"] = nmi_metadata["mapping_quality"].fillna("unknown")

print("Per-NMI metadata shape:", nmi_metadata.shape)
nmi_metadata.head()

Per-NMI metadata shape: (99, 9)


,NMI,building_codes,building_names,building_type,campus_codes,mapping_sources,mapping_quality,building_type_group,building_type_count
0,6102000812,<NA>,<NA>,Unknown,<NA>,LMS Serial to NMI Map; building_nmi_mapping_json,unmapped,Unknown,0
1,6102002302,298,"MELBOURNE THEATRE COMPANY HQ 252-264 STURT ST,...",Building - Minimal/Infra- Shed/Whse/Bulk Store...,COM,LMS Serial to NMI Map; building_nmi_mapping_json,one_building_mapped,Single,1
2,6102005454,110,THE SPOT,Building - Workspace/Teaching/Library,PAR,LMS Serial to NMI Map; Parkville Substation Ma...,one_building_mapped,Single,1
3,6102005592,110,THE SPOT,Building - Workspace/Teaching/Library,PAR,LMS Serial to NMI Map; Parkville Substation Ma...,one_building_mapped,Single,1
4,6102009742,278,"100 LEICESTER ST, CARLTON SOUTH, VIC 3053",Building - Workspace/Teaching/Library,PAR,LMS Serial to NMI Map; Parkville Substation Ma...,one_building_mapped,Single,1


## 3. Metric Functions

In [17]:
# General mathematical metric functions
def safe_divide(numerator, denominator):
    # Safe division to avoid zero denominators
    if denominator is None or pd.isna(denominator) or denominator == 0:
        return np.nan
    return numerator / denominator


def wape(y_true: pd.Series, y_pred: pd.Series) -> float:
    # Weighted Absolute Percentage Error. Suitable for comparing NMIs with different consumption scales
    valid = y_true.notna() & y_pred.notna()
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    denominator = y_true.abs().sum()
    if len(y_true) == 0 or denominator == 0:
        return np.nan
    return (y_true - y_pred).abs().sum() / denominator


def longest_true_run(mask: pd.Series) -> int:
    # Calculate the longest consecutive True run in a boolean mask
    if mask.empty or not mask.any():
        return 0
    groups = mask.ne(mask.shift()).cumsum()
    return int(mask.groupby(groups).sum().max())


def iqr_outlier_rate(values: pd.Series) -> float:
    # Calculate outlier rate using the IQR rule
    values = values.dropna()
    if len(values) < 10:
        return np.nan
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_mask = (values < lower) | (values > upper)
    return outlier_mask.mean()


def lag_corr(values: pd.Series, lag: int) -> float:
    # Calculate the correlation between y_t and y_(t-lag).
    values = values.astype(float)
    if values.notna().sum() <= lag + 10:
        return np.nan
    return values.corr(values.shift(lag))


def cycle_strength(active_df: pd.DataFrame, value_col: str, group_cols: list[str]) -> float:
    # Cycle strength: variance of group means divided by total variance. Larger values indicate stronger cyclic patterns.
    values = active_df[value_col].dropna()
    total_var = values.var()
    if len(values) < 10 or pd.isna(total_var) or total_var == 0:
        return np.nan
    grouped_mean = active_df.groupby(group_cols)[value_col].mean()
    return safe_divide(grouped_mean.var(), total_var)


def percentile_score(series: pd.Series, higher_is_better: bool) -> pd.Series:
    # Convert a metric into a 0-1 percentile score.
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    score = pd.Series(np.nan, index=s.index, dtype=float)
    valid = s.dropna()

    if valid.empty:
        return score
    if valid.nunique() == 1:
        score.loc[valid.index] = 0.5
        return score

    ranks = valid.rank(method="average", ascending=True, pct=True)
    if higher_is_better:
        score.loc[valid.index] = ranks
    else:
        # The minimum rank is close to 1/n and should receive the highest score, 1.
        score.loc[valid.index] = 1 - ranks + 1 / len(valid)

    return score.clip(0, 1)

In [18]:
# Baseline backtesting function

def baseline_backtest(active_df: pd.DataFrame, value_col: str) -> dict:
    """Run simple baseline backtesting within a single NMI active window.

    Baseline 1: lag_48, the same half-hour yesterday.
    Baseline 2: lag_336, the same half-hour last week.
    Baseline 3: calendar average, the expanding historical mean for the same weekday and half-hour.
    """
    df_bt = active_df[["date", value_col]].dropna().copy()
    df_bt = df_bt.sort_values("date")

    result = {
        "validation_start": pd.NaT,
        "validation_points": 0,
        "validation_months": 0,
        "WAPE_lag48": np.nan,
        "WAPE_lag336": np.nan,
        "WAPE_calendar": np.nan,
        "best_baseline_type": pd.NA,
        "best_baseline_WAPE": np.nan,
    }

    if df_bt.empty:
        return result

    active_days = (df_bt["date"].max() - df_bt["date"].min()).days
    if active_days < MIN_ACTIVE_DAYS_FOR_BACKTEST:
        return result

    # Use at least the first year of the active window as history; later points become validation.
    validation_start = df_bt["date"].min() + pd.Timedelta(days=MIN_ACTIVE_DAYS_FOR_BACKTEST)
    result["validation_start"] = validation_start

    y = df_bt[value_col].astype(float)

    # Seasonal naive predictions.
    pred_lag48 = y.shift(48)
    pred_lag336 = y.shift(336)

    # Calendar average prediction: use only past history from the same weekday and half-hour.
    df_bt["dow"] = df_bt["date"].dt.dayofweek
    df_bt["tod"] = df_bt["date"].dt.strftime("%H:%M")
    pred_calendar = y.groupby([df_bt["dow"], df_bt["tod"]]).transform(
        lambda s: s.shift(1).expanding(min_periods=4).mean()
    )

    validation_mask = df_bt["date"] >= validation_start
    result["validation_points"] = int(validation_mask.sum())
    result["validation_months"] = int(df_bt.loc[validation_mask, "date"].dt.to_period("M").nunique())

    if result["validation_points"] < MIN_VALIDATION_POINTS:
        return result

    y_valid = y[validation_mask]
    result["WAPE_lag48"] = wape(y_valid, pred_lag48[validation_mask])
    result["WAPE_lag336"] = wape(y_valid, pred_lag336[validation_mask])
    result["WAPE_calendar"] = wape(y_valid, pred_calendar[validation_mask])

    baseline_errors = {
        "lag48": result["WAPE_lag48"],
        "lag336": result["WAPE_lag336"],
        "calendar": result["WAPE_calendar"],
    }
    baseline_errors = {k: v for k, v in baseline_errors.items() if pd.notna(v)}

    if baseline_errors:
        best_type = min(baseline_errors, key=baseline_errors.get)
        result["best_baseline_type"] = best_type
        result["best_baseline_WAPE"] = baseline_errors[best_type]

    return result

## 4. Calculate Metrics for Each NMI

In [19]:
# Calculate all diagnostic metrics for each NMI

metric_rows = []

for nmi in nmi_cols:
    nmi_df = energy[["date", nmi]].copy()
    values_all = pd.to_numeric(nmi_df[nmi], errors="coerce")

    # Active window: the period between the first and last non-zero readings
    nonzero_mask = values_all.fillna(0).ne(0)

    if nonzero_mask.any():
        active_start = nmi_df.loc[nonzero_mask, "date"].min()
        active_end = nmi_df.loc[nonzero_mask, "date"].max()
    else:
        active_start = pd.NaT
        active_end = pd.NaT

    if pd.isna(active_start) or pd.isna(active_end):
        active_df = nmi_df.iloc[0:0].copy()
    else:
        active_df = nmi_df[nmi_df["date"].between(active_start, active_end)].copy()

    active_values = pd.to_numeric(active_df[nmi], errors="coerce")
    active_count = len(active_df)
    active_days = (active_end - active_start).days if pd.notna(active_start) else 0
    active_years = active_days / 365.25 if active_days else 0

    # Data quality metrics
    missing_rate = active_values.isna().mean() if active_count else np.nan
    zero_rate = active_values.fillna(0).eq(0).mean() if active_count else np.nan
    negative_count = int(active_values.lt(0).sum()) if active_count else 0
    longest_zero_run = longest_true_run(active_values.fillna(np.nan).eq(0)) if active_count else 0
    longest_zero_run_ratio = safe_divide(longest_zero_run, active_count)
    outlier_rate = iqr_outlier_rate(active_values)

    # Recent coverage: measures whether the NMI still has recent usable data
    recent_12m_start = latest_timestamp - pd.DateOffset(months=12)
    recent_24m_start = latest_timestamp - pd.DateOffset(months=24)

    recent_12m = nmi_df[nmi_df["date"].between(recent_12m_start, latest_timestamp)][nmi]
    recent_24m = nmi_df[nmi_df["date"].between(recent_24m_start, latest_timestamp)][nmi]

    recent_coverage_12m = recent_12m.fillna(0).ne(0).mean() if len(recent_12m) else np.nan
    recent_coverage_24m = recent_24m.fillna(0).ne(0).mean() if len(recent_24m) else np.nan

    # Usability status: follows the zero-rate-based status used in earlier EDA
    if pd.isna(zero_rate):
        status = "Dead"
    elif active_start == active_end:
        status = "Dead"
    elif zero_rate < 0.01:
        status = "Active"
    elif zero_rate < 0.10:
        status = "Mostly Active"
    elif zero_rate < 0.50:
        status = "Intermittent"
    elif zero_rate < 1.00:
        status = "Mostly inactive"
    else:
        status = "Dead"

    # Temporal dependence and seasonality metrics
    lag_1 = lag_corr(active_values, 1)
    lag_2 = lag_corr(active_values, 2)
    lag_48 = lag_corr(active_values, 48)
    lag_336 = lag_corr(active_values, 336)

    if not active_df.empty:
        active_df = active_df.assign(
            value=active_values.values,
            hour=active_df["date"].dt.hour,
            half_hour=active_df["date"].dt.strftime("%H:%M"),
            dayofweek=active_df["date"].dt.dayofweek,
            month=active_df["date"].dt.month,
        )
        daily_cycle_strength = cycle_strength(active_df, "value", ["half_hour"])
        weekly_pattern_strength = cycle_strength(active_df, "value", ["dayofweek", "half_hour"])
    else:
        daily_cycle_strength = np.nan
        weekly_pattern_strength = np.nan

    # Stability metrics: trend, yearly variation, and rolling shift based on daily totals
    if not active_df.empty:
        daily_total = active_df.set_index("date")["value"].resample("D").sum(min_count=1).dropna()
    else:
        daily_total = pd.Series(dtype=float)

    if len(daily_total) >= 30 and daily_total.mean() != 0:
        x = np.arange(len(daily_total))
        slope = np.polyfit(x, daily_total.values, 1)[0]
        trend_strength = abs(slope) / abs(daily_total.mean())
        rolling_mean = daily_total.rolling(window=30, min_periods=15).mean()
        structural_break_score = rolling_mean.diff().abs().max() / abs(daily_total.mean())
    else:
        trend_strength = np.nan
        structural_break_score = np.nan

    yearly_mean = daily_total.groupby(daily_total.index.year).mean() if len(daily_total) else pd.Series(dtype=float)
    yearly_variation = safe_divide(yearly_mean.std(), yearly_mean.mean()) if len(yearly_mean) >= 2 else np.nan

    monthly_mean = daily_total.groupby(daily_total.index.month).mean() if len(daily_total) else pd.Series(dtype=float)
    seasonality_strength = safe_divide(monthly_mean.var(), daily_total.var()) if len(monthly_mean) >= 2 and daily_total.var() != 0 else np.nan

    # Baseline backtesting error
    bt_input = active_df[["date", "value"]].rename(columns={"value": nmi}) if not active_df.empty else pd.DataFrame(columns=["date", nmi])
    bt = baseline_backtest(bt_input, nmi) if not bt_input.empty else {}

    metric_rows.append({
        "NMI": nmi,
        "active_start": active_start,
        "active_end": active_end,
        "active_years": active_years,
        "active_readings": active_count,
        "status": status,
        "missing_rate": missing_rate,
        "zero_rate": zero_rate,
        "negative_count": negative_count,
        "longest_zero_run": longest_zero_run,
        "longest_zero_run_hours": longest_zero_run * 0.5,
        "longest_zero_run_ratio": longest_zero_run_ratio,
        "outlier_rate": outlier_rate,
        "recent_coverage_12m": recent_coverage_12m,
        "recent_coverage_24m": recent_coverage_24m,
        "lag_1_corr": lag_1,
        "lag_2_corr": lag_2,
        "lag_48_corr": lag_48,
        "lag_336_corr": lag_336,
        "daily_cycle_strength": daily_cycle_strength,
        "weekly_pattern_strength": weekly_pattern_strength,
        "seasonality_strength": seasonality_strength,
        "trend_strength": trend_strength,
        "yearly_variation": yearly_variation,
        "structural_break_score": structural_break_score,
        **bt,
    })

metrics = pd.DataFrame(metric_rows)
print("Metrics table shape:", metrics.shape)
metrics.head()

Metrics table shape: (99, 33)


,NMI,active_start,active_end,active_years,active_readings,status,missing_rate,zero_rate,negative_count,longest_zero_run,longest_zero_run_hours,longest_zero_run_ratio,outlier_rate,recent_coverage_12m,recent_coverage_24m,lag_1_corr,lag_2_corr,lag_48_corr,lag_336_corr,daily_cycle_strength,weekly_pattern_strength,seasonality_strength,trend_strength,yearly_variation,structural_break_score,validation_start,validation_points,validation_months,WAPE_lag48,WAPE_lag336,WAPE_calendar,best_baseline_type,best_baseline_WAPE
0,6102000812,2013-01-01,2026-03-23 23:30:00,13.221081,231840,Active,0.0,0.000876,0,96,48.0,0.000414,0.024922,0.997260,0.998630,0.979415,0.960059,0.799482,0.869656,0.129226,0.276510,0.058688,0.000135,0.249968,0.043348,2014-01-01,214320,147,0.134564,0.119449,0.256382,lag336,0.119449
1,6102002302,2013-01-01,2026-03-23 23:30:00,13.221081,231840,Mostly Active,0.0,0.014864,0,96,48.0,0.000414,0.105055,0.979453,0.974259,0.977671,0.941799,0.690668,0.877202,0.297369,0.579896,0.072241,0.000132,0.238034,0.064987,2014-01-01,214320,147,0.327775,0.217085,0.303013,lag336,0.217085
2,6102005454,2013-01-01,2026-03-23 23:30:00,13.221081,231840,Active,0.0,0.000625,0,96,48.0,0.000414,0.026574,1.000000,1.000000,0.989394,0.970830,0.827727,0.921870,0.273324,0.430020,0.073567,0.000055,0.139207,0.050691,2014-01-01,214320,147,0.079524,0.059208,0.155487,lag336,0.059208
3,6102005592,2013-01-01,2026-03-23 23:30:00,13.221081,231840,Active,0.0,0.000824,0,96,48.0,0.000414,0.037401,1.000000,1.000000,0.967422,0.925170,0.743922,0.724098,0.327548,0.413785,0.249126,0.000097,0.263873,0.109400,2014-01-01,214320,147,0.364000,0.366622,0.515093,lag48,0.364000
4,6102009742,2013-01-01,2026-03-23 23:30:00,13.221081,231840,Active,0.0,0.000845,0,96,48.0,0.000414,0.015722,1.000000,1.000000,0.766304,0.599810,0.626953,0.684512,0.236868,0.330506,0.106601,0.000010,0.080232,0.041140,2014-01-01,214320,147,0.154464,0.143334,0.156296,lag336,0.143334


## 5. Merge Building Type Information

In [20]:
# Merge NMI metrics with building type information
summary = metrics.merge(nmi_metadata, on="NMI", how="left")

# Use Unknown for missing building metadata to avoid blank fields in the final table.
for col in ["building_codes", "building_names", "building_type", "building_type_group", "campus_codes", "mapping_sources", "mapping_quality"]:
    if col in summary.columns:
        summary[col] = summary[col].fillna("Unknown")

print("Summary with building metadata shape:", summary.shape)
summary[["NMI", "building_codes", "building_type", "building_type_group", "mapping_quality"]].head(10)

Summary with building metadata shape: (99, 41)


,NMI,building_codes,building_type,building_type_group,mapping_quality
0,6102000812,Unknown,Unknown,Unknown,unmapped
1,6102002302,298,Building - Minimal/Infra- Shed/Whse/Bulk Store...,Single,one_building_mapped
2,6102005454,110,Building - Workspace/Teaching/Library,Single,one_building_mapped
3,6102005592,110,Building - Workspace/Teaching/Library,Single,one_building_mapped
4,6102009742,278,Building - Workspace/Teaching/Library,Single,one_building_mapped
5,6102009743,278,Building - Workspace/Teaching/Library,Single,one_building_mapped
6,6102009744,278,Building - Workspace/Teaching/Library,Single,one_building_mapped
7,6102023971,102; 102|400|401|402|403|404|405; 400; 401; 40...,Building - Commercial; Building - Highly Servi...,Mixed,many_to_many
8,6102038376,151; 151|174|193; 174; 193,Building - Minimal/Infra- Shed/Whse/Bulk Store...,Mixed,many_to_one
9,6102046251,887,Building - Minimal/Infra- Shed/Whse/Bulk Store...,Single,one_building_mapped


## 6. Score Calculation

To avoid purely subjective judgement, use percentile-based scoring. Each metric is converted into a 0-1 score:

Overall score weights:

| Component | Weight | Interpretation |
|---|---:|---|
| `performance_score` | 35% | Whether simple baselines can forecast the NMI well |
| `data_quality_score` | 20% | Whether the raw series is complete and reliable |
| `history_score` | 15% | Whether the NMI has enough active and recent history |
| `temporal_pattern_score` | 15% | Whether the series has useful temporal structure |
| `stability_score` | 10% | Whether the series is stable over time |
| `mapping_score` | 5% | Whether the NMI-to-building relationship is clear |

`performance_score` receives the highest weight because it directly measures baseline forecast performance.

In [21]:
# Convert raw metrics into 0-1 scores
scored = summary.copy()

# Performance score: lower baseline WAPE is better
scored["performance_score"] = percentile_score(scored["best_baseline_WAPE"], higher_is_better=False)

# Data quality score: lower missing, zero, long-zero-run, and outlier rates are better
scored["missing_score"] = percentile_score(scored["missing_rate"], higher_is_better=False)
scored["zero_score"] = percentile_score(scored["zero_rate"], higher_is_better=False)
scored["zero_run_score"] = percentile_score(scored["longest_zero_run_ratio"], higher_is_better=False)
scored["outlier_score"] = percentile_score(scored["outlier_rate"], higher_is_better=False)
scored["data_quality_score"] = scored[["missing_score", "zero_score", "zero_run_score", "outlier_score"]].mean(axis=1, skipna=True)

# History score: longer active history and higher recent coverage are better
scored["active_years_score"] = percentile_score(scored["active_years"], higher_is_better=True)
scored["recent_coverage_score"] = percentile_score(scored["recent_coverage_24m"], higher_is_better=True)
scored["validation_months_score"] = percentile_score(scored["validation_months"], higher_is_better=True)
scored["history_score"] = scored[["active_years_score", "recent_coverage_score", "validation_months_score"]].mean(axis=1, skipna=True)

# Temporal pattern score: higher lag correlation and cycle strength are better
# Use abs(correlation), because strong linear dependence itself indicates predictable structure.
scored["lag_48_score"] = percentile_score(scored["lag_48_corr"].abs(), higher_is_better=True)
scored["lag_336_score"] = percentile_score(scored["lag_336_corr"].abs(), higher_is_better=True)
scored["daily_cycle_score"] = percentile_score(scored["daily_cycle_strength"], higher_is_better=True)
scored["weekly_pattern_score"] = percentile_score(scored["weekly_pattern_strength"], higher_is_better=True)
scored["seasonality_score"] = percentile_score(scored["seasonality_strength"], higher_is_better=True)
scored["temporal_pattern_score"] = scored[[
    "lag_48_score", "lag_336_score", "daily_cycle_score", "weekly_pattern_score", "seasonality_score"
]].mean(axis=1, skipna=True)

# Stability score: lower trend strength, yearly variation, and structural break score are better
scored["trend_score"] = percentile_score(scored["trend_strength"], higher_is_better=False)
scored["yearly_variation_score"] = percentile_score(scored["yearly_variation"], higher_is_better=False)
scored["structural_break_score_scaled"] = percentile_score(scored["structural_break_score"], higher_is_better=False)
scored["stability_score"] = scored[["trend_score", "yearly_variation_score", "structural_break_score_scaled"]].mean(axis=1, skipna=True)

# Mapping score: clearer mapping is better
mapping_score_map = {
    "one_building_mapped": 1.00,
    "many_to_one": 0.80,
    "multi_building_mapped": 0.65,
    "substation_shared_multi_building": 0.55,
    "many_to_many": 0.45,
    "unknown": 0.30,
    "unmapped": 0.20,
}
scored["mapping_score"] = scored["mapping_quality"].map(mapping_score_map).fillna(0.30)

print("Score columns created.")
scored[[
    "NMI", "performance_score", "data_quality_score", "history_score",
    "temporal_pattern_score", "stability_score", "mapping_score"
]].head()

Score columns created.


,NMI,performance_score,data_quality_score,history_score,temporal_pattern_score,stability_score,mapping_score
0,6102000812,0.623656,0.483380,0.574074,0.432653,0.418367,0.2
1,6102002302,0.139785,0.300634,0.545455,0.516327,0.312925,1.0
2,6102005454,0.935484,0.531282,0.727273,0.630612,0.581633,1.0
3,6102005592,0.043011,0.478046,0.727273,0.518367,0.255102,1.0
4,6102009742,0.451613,0.534323,0.727273,0.367347,0.823129,1.0


In [22]:
# Calculate the overall forecastability score
score_weights = {
    "performance_score": 0.35,
    "data_quality_score": 0.20,
    "history_score": 0.15,
    "temporal_pattern_score": 0.15,
    "stability_score": 0.10,
    "mapping_score": 0.05,
}

# Use a weighted average and automatically re-normalise weights when a component is missing.
def weighted_score(row: pd.Series, weights: dict[str, float]) -> float:
    numerator = 0.0
    denominator = 0.0
    for col, weight in weights.items():
        value = row.get(col)
        if pd.notna(value):
            numerator += weight * value
            denominator += weight
    return safe_divide(numerator, denominator)

scored["forecastability_score"] = scored.apply(lambda row: weighted_score(row, score_weights), axis=1)
scored["score_confidence"] = scored.apply(
    lambda row: sum(weight for col, weight in score_weights.items() if pd.notna(row.get(col))),
    axis=1,
)

scored[["NMI", "forecastability_score", "score_confidence"]].sort_values(
    "forecastability_score", ascending=False
).head(10)

,NMI,forecastability_score,score_confidence
29,6103005867,0.807161,1.0
73,VAAA000182,0.799451,1.0
38,6103022018,0.774199,1.0
35,6103015873,0.764166,1.0
7,6102023971,0.755301,1.0
2,6102005454,0.745522,1.0
68,VAAA000175,0.735488,1.0
74,VAAA000184,0.727436,1.0
77,VAAA000450,0.722307,1.0
27,6103002422,0.709015,1.0


## 7. Classification Rules

Because different NMIs have different active history lengths and operating periods, we do not force a single common training window. Instead, we identify each NMI's active window and construct a reproducible forecastability score using data quality, history coverage, temporal dependence, structural stability, and simple baseline backtesting error.

The final classification includes:

| Tier | Meaning |
|---|---|
| `Tier A - Strong forecasting candidate` | Good data quality, enough history, strong temporal pattern, and low baseline error |
| `Tier B - Usable with caution` | Usable for forecasting, but with some quality, stability, or mapping concerns |
| `Tier C - Short-history candidate` | Short history; better suited to pooled/global modelling or later evaluation |
| `Tier D - Difficult forecasting candidate` | High baseline error or unstable/noisy behaviour |
| `Exclude / Needs Review` | Insufficient or unreliable data; should be reviewed before individual forecasting |

`Tier C` does not mean the data is poor. It means the available history is too short to evaluate it in the same way as long-history NMIs.

In [23]:
# Forecastability tier classification
classified = scored.copy()
classified["forecastability_tier"] = pd.NA

# First flag NMIs that are clearly unsuitable for direct forecasting
exclude_mask = (
    classified["status"].isin(["Dead", "Mostly inactive"])
    | (classified["active_years"] < 0.25)
    | (classified["zero_rate"] >= 0.80)
)
classified.loc[exclude_mask, "forecastability_tier"] = "Exclude / Needs Review"

# Short history does not necessarily mean poor data: assign these NMIs to Tier C
short_history_mask = (
    classified["forecastability_tier"].isna()
    & (
        (classified["active_years"] < 1.0)
        | (classified["validation_months"].fillna(0) < 3)
        | (classified["best_baseline_WAPE"].isna())
    )
)
classified.loc[short_history_mask, "forecastability_tier"] = "Tier C - Short-history candidate"

# Classify eligible NMIs using score quantiles
eligible_mask = classified["forecastability_tier"].isna() & classified["forecastability_score"].notna()
eligible_scores = classified.loc[eligible_mask, "forecastability_score"]

if len(eligible_scores) >= 4:
    q25 = eligible_scores.quantile(0.25)
    q75 = eligible_scores.quantile(0.75)

    classified.loc[eligible_mask & (classified["forecastability_score"] >= q75), "forecastability_tier"] = (
        "Tier A - Strong forecasting candidate"
    )
    classified.loc[eligible_mask & (classified["forecastability_score"] <= q25), "forecastability_tier"] = (
        "Tier D - Difficult forecasting candidate"
    )
    classified.loc[classified["forecastability_tier"].isna() & eligible_mask, "forecastability_tier"] = (
        "Tier B - Usable with caution"
    )
else:
    classified.loc[eligible_mask, "forecastability_tier"] = "Tier B - Usable with caution"

classified["forecastability_tier"] = classified["forecastability_tier"].fillna("Exclude / Needs Review")

classified["forecastability_tier"].value_counts(dropna=False)

forecastability_tier
Tier B - Usable with caution                45
Tier D - Difficult forecasting candidate    23
Tier A - Strong forecasting candidate       23
Tier C - Short-history candidate             5
Exclude / Needs Review                       3
Name: count, dtype: int64

In [24]:
# Generate modelling recommendations and automatic reasons
def recommended_strategy(tier: str) -> str:
    # Return a recommended modelling strategy based on the tier
    if tier.startswith("Tier A"):
        return "Individual model; suitable for regular forecasting and benchmark model comparison"
    if tier.startswith("Tier B"):
        return "Individual model with caution; consider recent-window training, changepoint features, and anomaly handling"
    if tier.startswith("Tier C"):
        return "Pooled/global model; borrow strength from similar NMIs/building types and calendar/weather features"
    if tier.startswith("Tier D"):
        return "Low priority for individual model; consider aggregation or operational/data-quality review"
    return "Exclude from individual forecasting until mapping/data quality is reviewed"


def reason_text(row: pd.Series) -> str:
    # Automatically generate a reason string for reporting and interpretation
    reasons = []

    if row["active_years"] < 1:
        reasons.append("short active history")
    elif row["active_years"] >= 5:
        reasons.append("long active history")

    if pd.notna(row["best_baseline_WAPE"]):
        if row["performance_score"] >= 0.75:
            reasons.append("low baseline WAPE")
        elif row["performance_score"] <= 0.25:
            reasons.append("high baseline WAPE")
    else:
        reasons.append("insufficient backtesting history")

    if pd.notna(row["zero_rate"]):
        if row["zero_rate"] >= 0.50:
            reasons.append("high zero rate")
        elif row["zero_rate"] < 0.01:
            reasons.append("very low zero rate")

    if pd.notna(row["longest_zero_run_hours"]) and row["longest_zero_run_hours"] >= 24:
        reasons.append("long zero run")

    if pd.notna(row["lag_48_corr"]) or pd.notna(row["lag_336_corr"]):
        max_lag_corr = np.nanmax([abs(row.get("lag_48_corr", np.nan)), abs(row.get("lag_336_corr", np.nan))])
        if pd.notna(max_lag_corr) and max_lag_corr >= 0.80:
            reasons.append("strong daily/weekly temporal dependency")
        elif pd.notna(max_lag_corr) and max_lag_corr < 0.40:
            reasons.append("weak daily/weekly temporal dependency")

    if pd.notna(row["structural_break_score"]) and row["structural_break_score"] >= classified["structural_break_score"].quantile(0.75):
        reasons.append("large rolling-mean shift")

    if row.get("building_type_group") == "Mixed":
        reasons.append("multiple building types mapped")
    if row.get("mapping_quality") in {"unknown", "unmapped", "many_to_many"}:
        reasons.append("mapping quality needs review")

    if not reasons:
        reasons.append("balanced diagnostic profile")

    return "; ".join(reasons)


classified["recommended_model_strategy"] = classified["forecastability_tier"].apply(recommended_strategy)
classified["reason"] = classified.apply(reason_text, axis=1)

classified[["NMI", "forecastability_tier", "recommended_model_strategy", "reason"]].head(10)

,NMI,forecastability_tier,recommended_model_strategy,reason
0,6102000812,Tier B - Usable with caution,Individual model with caution; consider recent...,long active history; very low zero rate; long ...
1,6102002302,Tier D - Difficult forecasting candidate,Low priority for individual model; consider ag...,long active history; high baseline WAPE; long ...
2,6102005454,Tier A - Strong forecasting candidate,Individual model; suitable for regular forecas...,long active history; low baseline WAPE; very l...
3,6102005592,Tier D - Difficult forecasting candidate,Low priority for individual model; consider ag...,long active history; high baseline WAPE; very ...
4,6102009742,Tier B - Usable with caution,Individual model with caution; consider recent...,long active history; very low zero rate; long ...
5,6102009743,Tier B - Usable with caution,Individual model with caution; consider recent...,long active history; very low zero rate; long ...
6,6102009744,Tier B - Usable with caution,Individual model with caution; consider recent...,long active history; very low zero rate; long ...
7,6102023971,Tier A - Strong forecasting candidate,Individual model; suitable for regular forecas...,long active history; low baseline WAPE; very l...
8,6102038376,Tier B - Usable with caution,Individual model with caution; consider recent...,long active history; very low zero rate; long ...
9,6102046251,Tier D - Difficult forecasting candidate,Low priority for individual model; consider ag...,long active history; high baseline WAPE; very ...


## 8. Export the Final Table

The final table is saved in both CSV and Excel formats for reporting, visualisation, and downstream modelling.

In [25]:
# Prepare and export the final table
final_columns = [
    # Primary key and building metadata
    "NMI", "building_codes", "building_names", "building_type", "building_type_group",
    "building_type_count", "campus_codes", "mapping_quality", "mapping_sources",

    # Active window and coverage
    "active_start", "active_end", "active_years", "active_readings", "status",
    "recent_coverage_12m", "recent_coverage_24m",

    # Data quality
    "missing_rate", "zero_rate", "negative_count", "longest_zero_run",
    "longest_zero_run_hours", "longest_zero_run_ratio", "outlier_rate",

    # Temporal dependence and seasonality
    "lag_1_corr", "lag_2_corr", "lag_48_corr", "lag_336_corr",
    "daily_cycle_strength", "weekly_pattern_strength", "seasonality_strength",

    # Stability
    "trend_strength", "yearly_variation", "structural_break_score",

    # Baseline backtesting
    "validation_start", "validation_points", "validation_months",
    "WAPE_lag48", "WAPE_lag336", "WAPE_calendar",
    "best_baseline_type", "best_baseline_WAPE",

    # Component scores and overall score
    "performance_score", "data_quality_score", "history_score",
    "temporal_pattern_score", "stability_score", "mapping_score",
    "forecastability_score", "score_confidence",

    # Final classification and recommendation
    "forecastability_tier", "recommended_model_strategy", "reason",
]

# Guard against missing columns in unusual cases.
final_columns = [col for col in final_columns if col in classified.columns]

final_table = classified[final_columns].sort_values(
    ["forecastability_tier", "forecastability_score"],
    ascending=[True, False],
).reset_index(drop=True)

csv_output = OUTPUT_DIR / "NMI_forecastability_summary.csv"
xlsx_output = OUTPUT_DIR / "NMI_forecastability_summary.xlsx"

final_table.to_csv(csv_output, index=False)
final_table.to_excel(xlsx_output, index=False)

print("Saved CSV:", csv_output)
print("Saved Excel:", xlsx_output)
print("Final table shape:", final_table.shape)
final_table.head(20)

Saved CSV: /Users/chenlinchao/Desktop/Energy_Forecasting_Group_2/EDA_Tasks/NMI_score_classification_results/NMI_forecastability_summary.csv
Saved Excel: /Users/chenlinchao/Desktop/Energy_Forecasting_Group_2/EDA_Tasks/NMI_score_classification_results/NMI_forecastability_summary.xlsx
Final table shape: (99, 52)


,NMI,building_codes,building_names,building_type,building_type_group,building_type_count,campus_codes,mapping_quality,mapping_sources,active_start,active_end,active_years,active_readings,status,recent_coverage_12m,recent_coverage_24m,missing_rate,zero_rate,negative_count,longest_zero_run,longest_zero_run_hours,longest_zero_run_ratio,outlier_rate,lag_1_corr,lag_2_corr,lag_48_corr,lag_336_corr,daily_cycle_strength,weekly_pattern_strength,seasonality_strength,trend_strength,yearly_variation,structural_break_score,validation_start,validation_points,validation_months,WAPE_lag48,WAPE_lag336,WAPE_calendar,best_baseline_type,best_baseline_WAPE,performance_score,data_quality_score,history_score,temporal_pattern_score,stability_score,mapping_score,forecastability_score,score_confidence,forecastability_tier,recommended_model_strategy,reason
0,6103086897,734,CRESWICK GLASSHOUSE (LARGE),Building - Minimal/Infra- Shed/Whse/Bulk Store...,Single,1,CRE,one_building_mapped,LMS Serial to NMI Map; building_nmi_mapping_json,2025-07-30 08:30:00,2025-07-30 08:30:00,0.000000,1,Dead,0.000057,0.000029,0.0,0.000000,0,0,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,0,0,NaN,NaN,NaN,<NA>,NaN,NaN,0.816498,0.018519,NaN,NaN,1.00,0.540194,0.4,Exclude / Needs Review,Exclude from individual forecasting until mapp...,short active history; insufficient backtesting...
1,6102741362,245,"LEVELS 1-2, 32 LINCOLN SQ NORTH, CARLTON, VIC ...",Building - Workspace/Teaching/Library,Single,1,COM,one_building_mapped,LMS Serial to NMI Map; building_nmi_mapping_json,2014-01-01 00:00:00,2026-03-23 23:30:00,12.221766,214320,Mostly inactive,1.000000,1.000000,0.0,0.550952,0,96864,48432.0,0.451960,0.056929,0.986473,0.971335,0.890949,0.925464,0.027137,0.042341,0.009788,0.000067,1.301189,0.144187,2015-01-01,196800,135,0.275915,0.172578,1.747909,lag336,0.172578,0.311828,0.177206,0.505051,0.377551,0.221088,1.00,0.349080,1.0,Exclude / Needs Review,Exclude from individual forecasting until mapp...,long active history; high zero rate; long zero...
2,VAAA001271,245,"LEVELS 1-2, 32 LINCOLN SQ NORTH, CARLTON, VIC ...",Building - Workspace/Teaching/Library,Single,1,COM,one_building_mapped,LMS Serial to NMI Map; building_nmi_mapping_json,2014-01-01 00:00:00,2026-03-23 23:30:00,12.221766,214320,Mostly inactive,1.000000,1.000000,0.0,0.555879,0,96864,48432.0,0.451960,0.080921,0.975865,0.949506,0.880225,0.938207,0.105493,0.150400,0.014553,0.000248,1.034812,0.162930,2015-01-01,196800,135,0.279679,0.206224,1.311129,lag336,0.206224,0.182796,0.159374,0.505051,0.444898,0.061224,1.00,0.294468,1.0,Exclude / Needs Review,Exclude from individual forecasting until mapp...,long active history; high baseline WAPE; high ...
3,6103005867,144,KENNETH MYER BUILDING,Building - Highly Serviced / Wet Lab,Single,1,PAR,one_building_mapped,LMS Serial to NMI Map; Parkville Substation Ma...,2013-01-01 00:00:00,2026-03-23 23:30:00,13.221081,231840,Active,0.999829,0.999743,0.0,0.000746,0,96,48.0,0.000414,0.000772,0.981927,0.956693,0.764022,0.942580,0.537699,0.828488,0.038735,0.000003,0.048986,0.037872,2014-01-01,214320,147,0.084809,0.045530,0.068974,lag336,0.045530,0.956989,0.652597,0.614478,0.722449,0.911565,1.00,0.807161,1.0,Tier A - Strong forecasting candidate,Individual model; suitable for regular forecas...,long active history; low baseline WAPE; very l...
4,VAAA000182,139; 139|140|141|147|148|177|194; 140; 141; 14...,BABEL BUILDING; BAILLIEU LIBRARY; BIOSCIENCES ...,Building - Highly Serviced / Wet Lab; Building...,Mixed,2,PAR,many_to_one,LMS Serial to NMI Map; Parkville Substation Ma...,2013-01-01 00:00:00,2026-03-23 23:30:00,13.221081,231840,Active,1.000000,1.000000,0.0,0.000000,0,0,0.0,0.000000,0.018112,0.988760,0.969902,0.882553,0.904222,0.406374,0.487721,0.048587,0.000043,0.126781,0.029835,2014-01-01,214320,147,0.069154,0.064980,0.133833,lag336,0.064980,0.913978,0.739925,0.727273,0.683673,0.799320,0.80,0.799451,1.0,Tier A - Strong forecasting candidate,Individual model; suitable for regular foreca

In [26]:
# tier counts and building type distribution
print("Forecastability tier counts:")
display(final_table["forecastability_tier"].value_counts())

print("\nBuilding type group by tier:")
display(pd.crosstab(final_table["forecastability_tier"], final_table["building_type_group"], margins=True))

print("\nTop building types by count:")
display(final_table["building_type"].value_counts().head(20))

Forecastability tier counts:


forecastability_tier
Tier B - Usable with caution                45
Tier A - Strong forecasting candidate       23
Tier D - Difficult forecasting candidate    23
Tier C - Short-history candidate             5
Exclude / Needs Review                       3
Name: count, dtype: int64


Building type group by tier:


building_type_group,Mixed,Single,Unknown,All
forecastability_tier,,,,
Exclude / Needs Review,0,3,0,3
Tier A - Strong forecasting candidate,4,18,1,23
Tier B - Usable with caution,6,35,4,45
Tier C - Short-history candidate,0,4,1,5
Tier D - Difficult forecasting candidate,0,20,3,23
All,10,80,9,99



Top building types by count:


building_type
Building - Workspace/Teaching/Library                                                                                                                     51
Building - Minimal/Infra- Shed/Whse/Bulk Store/Spo                                                                                                        11
Unknown                                                                                                                                                    9
Building - Highly Serviced / Wet Lab                                                                                                                       8
Building - Residential                                                                                                                                     5
Building - Highly Serviced / Wet Lab; Building - Workspace/Teaching/Library                                                                                4
Building - Commercial                       